# ❤️ AI Heartbeat Reader (remote photoplethysmography / rPPG)
Reads heart rate from a face on video by detecting the tiny, invisible-to-the-eye color changes your skin undergoes with every heartbeat (blood volume pulse). This is real signal processing, not a simulated number.

**For a good reading, the video should have:**
- A steady, mostly-still face (handheld/shaky footage adds noise)
- Decent, consistent lighting on the face (no flickering light, no strong backlight)
- At least ~10-12 seconds of footage before the first BPM reading appears (the algorithm needs a window of data to find the periodic signal)

## Step 0: Ensure OpenCv install is complete (Colab sometimes ships a partial cv2)
Run this once before anything else. It's safe to always run. If you already ran later cells and hit `AttributeError: module 'cv2' has no attribute 'CascadeClassifier'`, run this cell, then go to **Runtime → Restart session**, then **Run all** again.

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install opencv-contrib-python-headless -q
print("Reinstalled OpenCV. Now go to Runtime -> Restart session, then Run all again.")

In [ ]:
!pip install scipy -q

In [ ]:
import cv2
import numpy as np
from scipy.signal import butter, filtfilt, detrend
from collections import deque
from google.colab import files
import os
import urllib.request

# Haar cascade face detector -- normally ships with OpenCV, but some Colab
# builds don't bundle the data files correctly, so fall back to downloading
# it directly from OpenCV's repo if the local one fails to load.
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    print("Local cascade file didn't load -- downloading it directly instead.")
    cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    cascade_path = "haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(cascade_url, cascade_path)
    face_cascade = cv2.CascadeClassifier(cascade_path)

assert not face_cascade.empty(), "Face cascade still failed to load -- check your internet/Colab environment."
print("Face detector loaded successfully.")

## Step 1: Upload your video

In [ ]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"FPS: {fps}, Resolution: {w}x{h}, Duration: {total_frames/fps:.1f}s")

## Step 2: Configuration

In [ ]:
# --- Signal window ---
WINDOW_SECONDS = 10          # how many seconds of history the BPM is computed from
BUFFER_LEN = int(fps * WINDOW_SECONDS)

# --- Valid heart-rate range (filters out noise outside plausible BPM) ---
MIN_BPM, MAX_BPM = 45, 180
LOWCUT_HZ = MIN_BPM / 60.0
HIGHCUT_HZ = MAX_BPM / 60.0

# --- How often to recompute the BPM (every N frames -- doesn't need to be every frame) ---
RECOMPUTE_EVERY_N_FRAMES = int(fps / 2)  # ~twice per second

# --- Smoothing for the displayed BPM number so it doesn't jitter ---
BPM_SMOOTHING = 0.85   # higher = smoother but slower to react to real changes

# --- Forehead ROI, as a fraction of the detected face box ---
FOREHEAD_X_RANGE = (0.30, 0.70)   # horizontal slice of the face box
FOREHEAD_Y_RANGE = (0.08, 0.25)   # vertical slice -- above eyebrows, below hairline

## Step 3: Signal processing functions

In [ ]:
def get_forehead_roi(frame, face_box):
    x, y, fw, fh = face_box
    x1 = x + int(fw * FOREHEAD_X_RANGE[0])
    x2 = x + int(fw * FOREHEAD_X_RANGE[1])
    y1 = y + int(fh * FOREHEAD_Y_RANGE[0])
    y2 = y + int(fh * FOREHEAD_Y_RANGE[1])
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
    return (x1, y1, x2, y2)

def mean_green_value(frame, roi):
    x1, y1, x2, y2 = roi
    if x2 <= x1 or y2 <= y1:
        return None
    region = frame[y1:y2, x1:x2]
    # green channel is the most sensitive to blood-volume changes (hemoglobin absorption)
    return float(np.mean(region[:, :, 1]))

def compute_bpm(green_buffer, fps):
    """Returns (bpm, filtered_signal) or (None, None) if the buffer isn't ready/valid."""
    if len(green_buffer) < BUFFER_LEN:
        return None, None

    signal = np.array(green_buffer)
    detrended = detrend(signal)

    nyq = fps / 2
    low = LOWCUT_HZ / nyq
    high = min(HIGHCUT_HZ / nyq, 0.99)  # keep below Nyquist
    try:
        b, a = butter(3, [low, high], btype="band")
        filtered = filtfilt(b, a, detrended)
    except ValueError:
        return None, None

    freqs = np.fft.rfftfreq(len(filtered), d=1 / fps)
    mag = np.abs(np.fft.rfft(filtered))

    mask = (freqs >= LOWCUT_HZ) & (freqs <= HIGHCUT_HZ)
    if not np.any(mask):
        return None, None

    peak_freq = freqs[mask][np.argmax(mag[mask])]
    bpm = peak_freq * 60
    return bpm, filtered

## Step 4: Drawing — status panel + live waveform

In [ ]:
def draw_waveform(frame, filtered_signal, panel_rect):
    x1, y1, x2, y2 = panel_rect
    cv2.rectangle(frame, (x1, y1), (x2, y2), (40, 40, 40), -1)

    if filtered_signal is None or len(filtered_signal) < 2:
        return

    recent = filtered_signal[-int(fps * 4):]  # last ~4 seconds of waveform
    norm = recent - np.min(recent)
    max_val = np.max(norm) if np.max(norm) > 0 else 1
    norm = norm / max_val

    panel_w = x2 - x1
    panel_h = y2 - y1
    points = []
    for i, val in enumerate(norm):
        px = x1 + int(i / max(1, len(norm) - 1) * panel_w)
        py = y2 - int(val * panel_h * 0.85) - int(panel_h * 0.075)
        points.append((px, py))

    for i in range(1, len(points)):
        cv2.line(frame, points[i - 1], points[i], (0, 60, 255), 2)

def draw_status_panel(frame, bpm_smoothed, buffer_fill_ratio, filtered_signal):
    panel_w, panel_h = 320, 200
    x, y = 10, 10
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + panel_w, y + panel_h), (0, 0, 0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

    cv2.putText(frame, "AI HEARTBEAT READER", (x + 15, y + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 60, 255), 2)

    if bpm_smoothed is None:
        pct = int(buffer_fill_ratio * 100)
        cv2.putText(frame, f"Reading signal... {pct}%", (x + 15, y + 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    else:
        cv2.putText(frame, f"{bpm_smoothed:.0f} BPM", (x + 15, y + 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 60, 255), 3)

    wave_rect = (x + 15, y + 100, x + panel_w - 15, y + panel_h - 15)
    draw_waveform(frame, filtered_signal, wave_rect)
    return frame

def draw_face_and_roi(frame, face_box, roi):
    fx, fy, fw, fh = face_box
    cv2.rectangle(frame, (fx, fy), (fx + fw, fy + fh), (0, 200, 255), 1)
    rx1, ry1, rx2, ry2 = roi
    cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), (0, 60, 255), 2)
    cv2.putText(frame, "forehead ROI", (rx1, ry1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 60, 255), 1)

## Step 5: Main processing loop

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
out = cv2.VideoWriter("heartbeat_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

green_buffer = deque(maxlen=BUFFER_LEN)
bpm_smoothed = None
last_filtered_signal = None
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray_for_detect = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_for_detect, scaleFactor=1.1,
                                            minNeighbors=6, minSize=(80, 80))

    if len(faces) > 0:
        # use the largest detected face if multiple are found
        face_box = max(faces, key=lambda f: f[2] * f[3])
        roi = get_forehead_roi(frame, face_box)
        g_val = mean_green_value(frame, roi)
        if g_val is not None:
            green_buffer.append(g_val)
        draw_face_and_roi(frame, face_box, roi)

    if frame_idx % RECOMPUTE_EVERY_N_FRAMES == 0:
        bpm_raw, filtered = compute_bpm(green_buffer, fps)
        if bpm_raw is not None:
            bpm_smoothed = bpm_raw if bpm_smoothed is None else \
                (BPM_SMOOTHING * bpm_smoothed + (1 - BPM_SMOOTHING) * bpm_raw)
            last_filtered_signal = filtered

    buffer_fill_ratio = len(green_buffer) / BUFFER_LEN
    frame = draw_status_panel(frame, bpm_smoothed, buffer_fill_ratio, last_filtered_signal)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print("Done -> heartbeat_output.mp4")
if bpm_smoothed is not None:
    print(f"Final reading: {bpm_smoothed:.0f} BPM")
else:
    print("No stable reading -- try a longer clip, steadier face, or better lighting.")

## Step 6: Save to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp heartbeat_output.mp4 /content/drive/MyDrive/heartbeat_output.mp4